In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
import os
from dotenv import load_dotenv

In [7]:
load_dotenv()
URL = os.getenv("SUPABASE_URL")
BUCKET_NAME = os.getenv("SUPABASE_BUCKET")

# 1. Load Preprocessed Data


In [8]:
cloud_path = f"{URL}/storage/v1/object/public/{BUCKET_NAME}/ulasan_ikd_preprocessed.csv"

df = pd.read_csv(cloud_path)

df = df.fillna("")

print("Data loaded successfully. Shape:", df.shape)

Data loaded successfully. Shape: (34902, 7)


# 2. Setup Variables



In [9]:
y = df['label']
features = ['text_baseline', 'text_standard', 'text_custom']

idx_temp, idx_test = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_score = 0
best_dataset = ""

# 3. Ablation Study Loop

In [10]:
print("--- MEMULAI ABLATION STUDY (5-Fold CV pada Data Train+Val) ---\n")

for col in features:
    X_temp = df.loc[idx_temp, col]
    y_temp = y.loc[idx_temp]
    
    # Build Pipeline for TF-IDF Vectorization with SVM Model
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("svm", SVC(kernel="linear", random_state=42))
    ])
    
    # Evaluating the Pipeline
    scores = cross_val_score(pipe, X_temp, y_temp, cv=cv, scoring="f1_macro", n_jobs=-1)
    mean_f1 = scores.mean()
    std_f1 = scores.std()
    
    print(f"Dataset : {col}")
    print(f"Macro F1: {mean_f1:.4f} (+/- {std_f1:.4f})")
    print("-" * 35)
    
    if mean_f1 > best_score:
        best_score = mean_f1
        best_dataset = col

print(f"\nPemenang Uji Ablasi adalah {best_dataset.upper()} dengan rata-rata CV Macro F1 {best_score:.4f}")
print("Gunakan dataset ini untuk tahapan Optuna dan LSTM selanjutnya.")

--- MEMULAI ABLATION STUDY (5-Fold CV pada Data Train+Val) ---

Dataset : text_baseline
Macro F1: 0.9237 (+/- 0.0027)
-----------------------------------
Dataset : text_standard
Macro F1: 0.9150 (+/- 0.0042)
-----------------------------------
Dataset : text_custom
Macro F1: 0.9204 (+/- 0.0030)
-----------------------------------

Pemenang Uji Ablasi adalah TEXT_BASELINE dengan rata-rata CV Macro F1 0.9237
Gunakan dataset ini untuk tahapan Optuna dan LSTM selanjutnya.


Pengujian ablasi preprocessing menggunakan Stratified 5-Fold Cross-Validation pada data latih menunjukkan hasil yang berlawanan dengan asumsi umum pemrosesan teks tradisional. 

Dataset `text_baseline` (tanpa penghapusan stopword) memperoleh rata-rata Macro F1-Score tertinggi sebesar 0.9237 (± 0.0027). Penggunaan kamus stopword standar (`text_standard`) justru mendegradasi performa model ke 0.9150 (± 0.0042) karena hilangnya kata-kata pembawa konteks dan negasi. Meskipun modifikasi penghapusan stopword (`text_custom`) berhasil memulihkan sebagian performa menjadi 0.9204 (± 0.0030), skor tersebut tetap tidak dapat melampaui baseline. Hal ini membuktikan bahwa pada teks ulasan berukuran pendek, pembobotan TF-IDF secara inheren telah mampu meredam kata-kata umum lebih baik daripada penghapusan kamus statis. Berdasarkan temuan empiris ini, penelitian dilanjutkan menggunakan `text_baseline`.